# Sparse-1M Streaming Benchmark

This notebook benchmarks 5 algorithms (`linscan`, `cufe`, `shnsw`, `binary-splitting`, `double-group-testing`) on the `sparse-1M` dataset in a streaming setting.

The streaming process consists of:
1.  **Step 1**: Insert 500,000 points.
2.  **Step 2**: Insert 100,000 points (Total 600k).
3.  **Step 3**: Insert 200,000 points (Total 800k).
4.  **Step 4**: Insert 200,000 points (Total 1M).

We will verify their performance (Recall) after each step.

In [1]:
import os
import yaml
import subprocess
import matplotlib.pyplot as plt
import numpy as np
import h5py

# Define the Runbook
runbook = {
    "sparse-1M": {
        "max_pts": 1000000,
        1: {"operation": "insert", "start": 0, "end": 500000},
        2: {"operation": "search"},
        3: {"operation": "insert", "start": 500000, "end": 600000},
        4: {"operation": "search"},
        5: {"operation": "insert", "start": 600000, "end": 800000},
        6: {"operation": "search"},
        7: {"operation": "insert", "start": 800000, "end": 1000000},
        8: {"operation": "search"}
    }
}

runbook_path = "sparse_streaming_runbook.yaml"
with open(runbook_path, "w") as f:
    yaml.dump(runbook, f)

print(f"Created runbook: {runbook_path}")

Created runbook: sparse_streaming_runbook.yaml


In [2]:
# 2. Generate Ground Truth for this runbook
# This is required for the benchmark runner to calculate recall 'on the fly' or post-hoc.
print("Generating Ground Truth...")
gt_cmd = [
    "python3", "-m", "benchmark.streaming.compute_gt",
    "--dataset", "sparse-1M",
    "--runbook_file", runbook_path
]
subprocess.run(gt_cmd, check=True)
print("Ground Truth Generation Complete.")

Generating Ground Truth...


Computing GT internally for step 2 with 500000 points...
data/sparse/queries.dev.csr.gz
Written data/sparse/1000000/sparse_streaming_runbook.yaml/step2.gt100
Computing GT internally for step 4 with 600000 points...
data/sparse/queries.dev.csr.gz
Written data/sparse/1000000/sparse_streaming_runbook.yaml/step4.gt100
Computing GT internally for step 6 with 800000 points...
data/sparse/queries.dev.csr.gz
Written data/sparse/1000000/sparse_streaming_runbook.yaml/step6.gt100
Computing GT internally for step 8 with 1000000 points...
data/sparse/queries.dev.csr.gz
Written data/sparse/1000000/sparse_streaming_runbook.yaml/step8.gt100
Ground Truth Generation Complete.


In [ ]:
# 3. Running Benchmarks
algorithms = ["linscan", "cufe", "shnsw", "binary-splitting", "double-group-testing"]
# algorithms = ["linscan"] # Uncomment to test just one first

for algo in algorithms:
    print(f"Running benchmark for {algo}...")
    cmd = [
        "python3", "-u", "run.py",
        "--dataset", "sparse-1M",
        "--algorithm", algo,
        "--neurips23track", "sparse",
        "--runbook_path", runbook_path,
        "--nodocker",
        "--count", "10", # k=10,
        "--definitions", "sparse_streaming_definitions.yaml"
    ]
    try:
        subprocess.run(cmd, check=True)
        print(f"Finished {algo}")
    except subprocess.CalledProcessError as e:
        print(f"Failed {algo}: {e}")


Running benchmark for linscan...


unzipped version of file data/sparse/queries.dev.csr.gz already exists
file data/sparse/base_1M.dev.gt already exists
unzipped version of file data/sparse/queries.hidden.csr.gz already exists
file data/sparse/base_full.hidden.gt already exists
unzipped version of file data/sparse/base_1M.csr.gz already exists
2026-01-18 16:54:29,276 - annb - INFO - running only linscan
2026-01-18 16:54:29,276 - annb - INFO - Order: [Definition(algorithm='linscan', constructor='Linscan', module='neurips23.sparse.linscan.linscan', docker_tag='neurips23-sparse-linscan', docker_volumes=[], arguments=['ip', {}], query_argument_groups=[[{'budget': 0.5}], [{'budget': 1}], [{'budget': 2}], [{'budget': 4}], [{'budget': 5}], [{'budget': 6}], [{'budget': 7}], [{'budget': 8}], [{'budget': 10}]], disabled=False)]
RW Namespace(dataset='sparse-1M', dataset_path=None, count=10, definitions='algos-2021.yaml', algorithm='linscan', docker_tag=None, list_algorithms=False, force=False, rebuild=False, runs=5, timeout=43200,

Process Process-1:
Traceback (most recent call last):
  File "/home/tejassharma/miniconda3/envs/benchmarks/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/tejassharma/miniconda3/envs/benchmarks/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/tejassharma/big-ann-benchmarks/benchmark/main.py", line 50, in run_worker
    run_no_docker(definition, args.dataset, args.count,
  File "/home/tejassharma/big-ann-benchmarks/benchmark/runner.py", line 379, in run_no_docker
    run_from_cmdline(cmd)
  File "/home/tejassharma/big-ann-benchmarks/benchmark/runner.py", line 228, in run_from_cmdline
    run(definition, args.dataset, args.count, args.runs, args.rebuild,
  File "/home/tejassharma/big-ann-benchmarks/benchmark/runner.py", line 101, in run
    algo.set_query_arguments(*query_arguments)
  File "/home/tejassharma/big-ann-benchmarks/neurips23/sparse/shnsw/shnsw.py", line 55,

Failed shnsw: Command '['python3', '-u', 'run.py', '--dataset', 'sparse-1M', '--algorithm', 'shnsw', '--neurips23track', 'sparse', '--runbook_path', 'sparse_streaming_runbook.yaml', '--nodocker', '--count', '10']' returned non-zero exit status 1.
Running benchmark for binary-splitting...
unzipped version of file data/sparse/queries.dev.csr.gz already exists
file data/sparse/base_1M.dev.gt already exists
unzipped version of file data/sparse/queries.hidden.csr.gz already exists
file data/sparse/base_full.hidden.gt already exists
unzipped version of file data/sparse/base_1M.csr.gz already exists
2026-01-18 17:06:52,365 - annb - INFO - running only binary-splitting
Failed binary-splitting: Command '['python3', '-u', 'run.py', '--dataset', 'sparse-1M', '--algorithm', 'binary-splitting', '--neurips23track', 'sparse', '--runbook_path', 'sparse_streaming_runbook.yaml', '--nodocker', '--count', '10']' returned non-zero exit status 1.
Running benchmark for double-group-testing...


Traceback (most recent call last):
  File "/home/tejassharma/big-ann-benchmarks/run.py", line 6, in <module>
    main()
  File "/home/tejassharma/big-ann-benchmarks/benchmark/main.py", line 265, in main
    raise Exception('Nothing to run')
Exception: Nothing to run


unzipped version of file data/sparse/queries.dev.csr.gz already exists
file data/sparse/base_1M.dev.gt already exists
unzipped version of file data/sparse/queries.hidden.csr.gz already exists
file data/sparse/base_full.hidden.gt already exists
unzipped version of file data/sparse/base_1M.csr.gz already exists
2026-01-18 17:06:53,217 - annb - INFO - running only double-group-testing
Failed double-group-testing: Command '['python3', '-u', 'run.py', '--dataset', 'sparse-1M', '--algorithm', 'double-group-testing', '--neurips23track', 'sparse', '--runbook_path', 'sparse_streaming_runbook.yaml', '--nodocker', '--count', '10']' returned non-zero exit status 1.


Traceback (most recent call last):
  File "/home/tejassharma/big-ann-benchmarks/run.py", line 6, in <module>
    main()
  File "/home/tejassharma/big-ann-benchmarks/benchmark/main.py", line 265, in main
    raise Exception('Nothing to run')
Exception: Nothing to run


In [ ]:
# 4. Analyze and Plot Results
# Find the result files
result_dir = f"results/neurips23/streaming/{os.path.split(runbook_path)[1]}/sparse-1M/10"

data = {}

for algo in algorithms:
    algo_dir = os.path.join(result_dir, algo)
    if not os.path.exists(algo_dir):
        print(f"No results found for {algo}")
        continue
    
    # Iterate through HDF5 files (there might be multiple runs/params, we pick one for demo)
    for f_name in os.listdir(algo_dir):
        if f_name.endswith(".hdf5"):
            f_path = os.path.join(algo_dir, f_name)
            try:
                with h5py.File(f_path, "r") as f:
                    # Depending on how streaming results are stored 
                    # Usually main metrics are in attributes or dedicated datasets per step
                    # Let's inspect ONE file structure if possible, 
                    # but typically standard runner stores 'recall' if computed.
                    # Or we may need to compute recall using plotting utils.
                    # For now, let's look for 'recall' in attributes or simple output.
                    
                    # Assuming plot.py computing logic:
                    # We might not have computed recall inside the HDF5 yet if we just ran 'run.py'.
                    # 'run.py' stores neighbors. 'plot.py' computes metrics.
                    pass
            except Exception as e:
                print(f"Error reading {f_path}: {e}")

print("To visualize the results properly, please run:")
print(f"python3 plot.py --dataset sparse-1M --neurips23track streaming --runbook_path {runbook_path}")


No results found for linscan
No results found for cufe
No results found for shnsw
No results found for binary-splitting
No results found for double-group-testing
To visualize the results properly, please run:
python3 plot.py --dataset sparse-1M --neurips23track streaming --runbook_path sparse_streaming_runbook.yaml
